<a href="https://colab.research.google.com/github/musab855/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import duckdb
from google.colab import userdata
import pandas as pd
import numpy as np

HF_TOKEN = userdata.get('HF_TOKEN2')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

feature_frame = con.sql(f"""
WITH daily_averages AS (
  SELECT
    content_hash_id,
    client_hash_id,
    AVG(CASE WHEN report_date < '2026-03-16' AND gsc_data_available THEN gsc_impressions ELSE NULL END) as gsc_impressions_1to15,
    AVG(CASE WHEN report_date < '2026-03-16' AND gsc_data_available THEN gsc_clicks ELSE NULL END) as gsc_clicks_1to15,
    AVG(CASE WHEN report_date < '2026-03-16' AND gsc_data_available THEN gsc_avg_position ELSE NULL END) as position_1to15,
    AVG(CASE WHEN report_date >= '2026-03-16' AND gsc_data_available THEN gsc_impressions ELSE NULL END) as gsc_impressions_16to31,
    AVG(CASE WHEN report_date >= '2026-03-16' AND gsc_data_available THEN gsc_clicks ELSE NULL END) as gsc_clicks_16to31,
    AVG(CASE WHEN report_date >= '2026-03-16' AND gsc_data_available THEN gsc_avg_position ELSE NULL END) as position_16to31
  FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
  WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01' AND gsc_data_available = TRUE
  GROUP BY content_hash_id, client_hash_id
)
SELECT
  content_hash_id AS content_id,
  gsc_impressions_1to15 as gsc_impressions_avg,
  gsc_clicks_1to15 as gsc_clicks_avg,
  position_1to15 as position_avg,
  (CASE WHEN gsc_impressions_16to31 < gsc_impressions_1to15 THEN 1 ELSE 0 END) as is_declining_label
FROM daily_averages
""")

df = feature_frame.df()

print(f"Loaded {len(df):,} rows")
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded 176,738 rows


,content_id,gsc_impressions_avg,gsc_clicks_avg,position_avg,is_declining_label
0,content_b7e512995f79d5a6,28.600000,0.133333,4.247255,0
1,content_a7da352b73b02668,162.666667,0.533333,7.259861,1
2,content_d056587ff7faca0c,85.333333,0.600000,4.468441,0
3,content_bfd1e41c2af250c8,1.583333,0.000000,9.208333,0
4,content_2662845f598544ef,6.466667,0.000000,8.765983,1


In [2]:
no_position_data = (df['position_avg'] == 0).sum()
print(f"Excluding {no_position_data} rows with position_avg == 0 (no GSC position data, not rank 1)")

df = df[df['position_avg'] > 0].copy()
print(f"Remaining rows: {len(df):,}")

Excluding 1306 rows with position_avg == 0 (no GSC position data, not rank 1)
Remaining rows: 150,675


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Impressions verdict: CONFIRMED — decline rate rises across every bucket
(41.6% to 54.4% to 56.3% to 61.8%). Biggest jump is actually between <100
and 100-500 (+12.8 pts), not at 500, but the trend is monotonic so 500
still holds as a "meaningful volume" cutoff. Ties to Quick Win flag.

Position verdict: MIXED — decline rate barely moves (44.0% to 43.7% to
44.3%) then drops at 30+ (40.1%), so it's not monotonic and the jump at
11 isn't real in this data. Kept as a threshold anyway because it reflects
a reasonable business assumption (off page one = under-optimized), not
because the raw numbers confirm it. Ties to CTR Fix flag, weakly.

In [3]:
imp_bins = [0, 100, 500, 2000, np.inf]
imp_labels = ['<100', '100-500', '500-2000', '2000+']
imp_table = (
    df.assign(bucket=pd.cut(df['gsc_impressions_avg'], bins=imp_bins, labels=imp_labels, right=False))
      .groupby('bucket', observed=True)['is_declining_label']
      .agg(n='count', pct_declining='mean')
)
imp_table['pct_declining'] = (imp_table['pct_declining'] * 100).round(1)
print("Impressions buckets:")
display(imp_table)

pos_bins = [0, 4, 11, 31, np.inf]
pos_labels = ['1-3', '4-10', '11-30', '30+']
pos_table = (
    df.assign(bucket=pd.cut(df['position_avg'], bins=pos_bins, labels=pos_labels, right=False))
      .groupby('bucket', observed=True)['is_declining_label']
      .agg(n='count', pct_declining='mean')
)
pos_table['pct_declining'] = (pos_table['pct_declining'] * 100).round(1)
print("Position buckets:")
display(pos_table)

Impressions buckets:


,n,pct_declining
bucket,,
<100,129909,41.6
100-500,17661,54.4
500-2000,2919,56.3
2000+,186,61.8


Position buckets:


,n,pct_declining
bucket,,
1-3,26551,44.0
4-10,63528,43.7
11-30,38026,44.3
30+,22570,40.1


In [4]:
IMPRESSIONS_THRESHOLD = 500
POSITION_THRESHOLD = 11

REASON_CODES = {
    "HIGH_VOLUME_POOR_POSITION": "impressions_avg >= 500 and position_avg >= 11"
}

print("Rule thresholds:")
print(f"  impressions_avg >= {IMPRESSIONS_THRESHOLD}")
print(f"  position_avg >= {POSITION_THRESHOLD}")
print("\nReason codes:")
for code, condition in REASON_CODES.items():
    print(f"  {code}: {condition}")

Rule thresholds:
  impressions_avg >= 500
  position_avg >= 11

Reason codes:
  HIGH_VOLUME_POOR_POSITION: impressions_avg >= 500 and position_avg >= 11


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

**Scoring:** Within the flagged pages, rank by `impressions_avg` — higher volume
means more traffic on the table, so those pages get reviewed first.

In [5]:
# Assumes `df` is your loaded feature frame with columns:
# content_id, gsc_impressions_avg, gsc_clicks_avg, position_avg, is_declining_label

flagged = df[
    (df['gsc_impressions_avg'] >= IMPRESSIONS_THRESHOLD) &
    (df['position_avg'] >= POSITION_THRESHOLD)
].copy()

flagged['action_score'] = flagged['gsc_impressions_avg']
flagged['reason_code'] = 'HIGH_VOLUME_POOR_POSITION'
flagged['action_label'] = 'Refresh Opportunity'

ranked_queue = flagged.sort_values('action_score', ascending=False).reset_index(drop=True)

print(f"Flagged {len(ranked_queue):,} of {len(df):,} pages ({len(ranked_queue)/len(df)*100:.1f}%)")
ranked_queue[['content_id', 'gsc_impressions_avg', 'position_avg',
              'action_score', 'reason_code', 'action_label']].head(10)

Flagged 1,074 of 150,675 pages (0.7%)


,content_id,gsc_impressions_avg,position_avg,action_score,reason_code,action_label
0,content_e8a52cf3d5988c07,9544.866667,16.018687,9544.866667,HIGH_VOLUME_POOR_POSITION,Refresh Opportunity
1,content_36e53e9c707674fc,7327.266667,33.354423,7327.266667,HIGH_VOLUME_POOR_POSITION,Refresh Opportunity
2,content_3df3f32f3fd58dea,5602.733333,24.501281,5602.733333,HIGH_VOLUME_POOR_POSITION,Refresh Opportunity
3,content_5e1c049f62e33b11,4862.666667,18.233335,4862.666667,HIGH_VOLUME_POOR_POSITION,Refresh Opportunity
4,content_82e35c4845e6c391,4677.933333,18.269589,4677.933333,HIGH_VOLUME_POOR_POSITION,Refresh Opportunity
5,content_df47d1b976106de4,4422.800000,25.221991,4422.800000,HIGH_VOLUME_POOR_POSITION,Refresh Opportunity
6,content_559cdd76da9306de,4161.200000,37.866104,4161.200000,HIGH_VOLUME_POOR_POSITION,Refresh Opportunity
7,content_bdf60c86117079be,4018.466667,31.046299,4018.466667,HIGH_VOLUME_POOR_POSITION,Refresh Opportunity
8,content_9fff53e827550f9d,3764.666667,22.357092,3764.666667,HIGH_VOLUME_POOR_POSITION,Refresh Opportunity
9,content_573804af4f4fa09f,3529.866667,29.926699,3529.866667,HIGH_VOLUME_POOR_POSITION,Refresh Opportunity


In [6]:
import os
os.makedirs('work/outputs', exist_ok=True)

output_cols = ['content_id', 'gsc_impressions_avg', 'position_avg',
               'action_score', 'reason_code', 'action_label']

out_path = 'work/outputs/baseline_action_score.csv'
ranked_queue[output_cols].to_csv(out_path, index=False)

print(f"Wrote {len(ranked_queue):,} rows to {out_path}")

Wrote 1,074 rows to work/outputs/baseline_action_score.csv


In [7]:
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = df['is_declining_label'].mean()
print(f"Base rate (overall decline %): {base_rate:.3f}")

for k in [10, 20, 50]:
    p = precision_at_k(ranked_queue['action_score'].values, ranked_queue['is_declining_label'].values, k)
    print(f"Precision@{k}: {p:.3f}  (vs base rate {base_rate:.3f})")

Base rate (overall decline %): 0.434
Precision@10: 1.000  (vs base rate 0.434)
Precision@20: 0.950  (vs base rate 0.434)
Precision@50: 0.880  (vs base rate 0.434)


In [8]:
import json

metrics = {
    "rule": "HIGH_VOLUME_POOR_POSITION",
    "impressions_threshold": IMPRESSIONS_THRESHOLD,
    "position_threshold": POSITION_THRESHOLD,
    "total_pages": len(df),
    "flagged_pages": len(ranked_queue),
    "flagged_pct": round(len(ranked_queue) / len(df) * 100, 2),
    "base_rate": round(base_rate, 3),
    "precision_at_10": round(precision_at_k(ranked_queue['action_score'].values, ranked_queue['is_declining_label'].values, 10), 3),
    "precision_at_20": round(precision_at_k(ranked_queue['action_score'].values, ranked_queue['is_declining_label'].values, 20), 3),
    "precision_at_50": round(precision_at_k(ranked_queue['action_score'].values, ranked_queue['is_declining_label'].values, 50), 3),
    "impressions_verdict": "CONFIRMED",
    "position_verdict": "MIXED",
    "excluded_no_position_data_rows": int(no_position_data),
}

with open('work/outputs/baseline_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print("Wrote work/outputs/baseline_metrics.json")

Wrote work/outputs/baseline_metrics.json


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

**Confidence logic:** High = far past both thresholds (position 20+, meaning well
off page one, AND impressions 2000+, meaning a lot of traffic at stake — little
room for noise to flip the call). Medium = past both thresholds but not by much.
Low = within a small margin of either threshold, where a single noisy day of data
could put the page back on the other side of the line.

**Wrong_if:** generated per row from its own numbers below — a borderline position
gets flagged for noise risk, a high-impression/near-zero-click row gets flagged
for possible bot traffic, and a clean, comfortably-past-threshold row gets the
more general "position genuinely recovers" caveat.

In [9]:
top20 = ranked_queue.head(20)[['content_id', 'gsc_impressions_avg', 'gsc_clicks_avg',
                                'position_avg', 'action_score', 'reason_code',
                                'action_label']].copy()

def confidence_note(row):
    if row['position_avg'] >= 20 and row['gsc_impressions_avg'] >= 2000:
        return 'High'
    elif row['position_avg'] < POSITION_THRESHOLD + 1 or row['gsc_impressions_avg'] < IMPRESSIONS_THRESHOLD + 100:
        return 'Low'
    return 'Med'

def wrong_if(row):
    ctr = row['gsc_clicks_avg'] / row['gsc_impressions_avg'] if row['gsc_impressions_avg'] > 0 else 0
    reasons = []
    if row['position_avg'] < POSITION_THRESHOLD + 1:
        reasons.append(f"position ({row['position_avg']:.1f}) is barely past the {POSITION_THRESHOLD} "
                        f"cutoff — one noisy week of data could put it back below the line")
    if ctr < 0.001:
        reasons.append(f"CTR is near zero ({ctr:.4f}) despite {row['gsc_impressions_avg']:.0f} impressions "
                        f"— worth checking this isn't bot/scraper traffic")
    if not reasons:
        reasons.append("position genuinely recovers on its own next month, or the page was already "
                        "refreshed recently and this data hasn't caught up yet")
    return "; ".join(reasons)

top20['confidence'] = top20.apply(confidence_note, axis=1)
top20['wrong_if'] = top20.apply(wrong_if, axis=1)

for i, row in top20.iterrows():
    print(f"Row {i+1}: action={row['action_label']} | reason={row['reason_code']} "
          f"| confidence={row['confidence']} | wrong_if: {row['wrong_if']}")

Row 1: action=Refresh Opportunity | reason=HIGH_VOLUME_POOR_POSITION | confidence=Med | wrong_if: position genuinely recovers on its own next month, or the page was already refreshed recently and this data hasn't caught up yet
Row 2: action=Refresh Opportunity | reason=HIGH_VOLUME_POOR_POSITION | confidence=High | wrong_if: position genuinely recovers on its own next month, or the page was already refreshed recently and this data hasn't caught up yet
Row 3: action=Refresh Opportunity | reason=HIGH_VOLUME_POOR_POSITION | confidence=High | wrong_if: position genuinely recovers on its own next month, or the page was already refreshed recently and this data hasn't caught up yet
Row 4: action=Refresh Opportunity | reason=HIGH_VOLUME_POOR_POSITION | confidence=Med | wrong_if: position genuinely recovers on its own next month, or the page was already refreshed recently and this data hasn't caught up yet
Row 5: action=Refresh Opportunity | reason=HIGH_VOLUME_POOR_POSITION | confidence=Med | wr

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: 12 of the top 20 rows (60%) show CTR below 0.001 — essentially zero
clicks despite impressions in the 3,000-9,500 range. This is a red flag, not
just a caveat: real search demand at position 16-38 typically still generates
some clicks. These rows may reflect bot traffic, indexing artifacts, or pages
that are technically "seen" but never genuinely surfaced to users. Ranking by
impressions_avg alone doesn't catch this - a future version of the score should
probably weight in CTR or clicks_avg, not just raw impressions.

Leakage check: This rule uses only gsc_impressions_avg and position_avg from
the days 1-15 feature window. It does not use trend_pct, is_declining_label,
any days 16-31 data, or any product flags as inputs. is_declining_label is
referenced only afterward, to measure precision, never inside the rule itself.



Precision: 1.000 / 0.950 / 0.880 at K=10/20/50, against a 0.434 base rate —
roughly double random selection at every cut, and ahead of Musab's Week-2
benchmark (Precision@20 = 0.900). Strong performance despite the CTR anomaly
noted above; the bot-traffic-looking rows still mostly carry the correct
label, so they're not hurting precision even though they look suspicious.

In [10]:
def flag_weak(row):
    flags = []
    if row['position_avg'] < POSITION_THRESHOLD + 0.5:
        flags.append('borderline position (near threshold)')
    if row['gsc_impressions_avg'] > 0:
        ctr = row['gsc_clicks_avg'] / row['gsc_impressions_avg']
        if ctr < 0.001:
            flags.append('near-zero CTR despite high impressions (possible bot traffic)')
    return '; '.join(flags) if flags else 'no flag'

top20['weak_pick_flag'] = top20.apply(flag_weak, axis=1)
display(top20[['content_id', 'gsc_impressions_avg', 'position_avg', 'weak_pick_flag']])

rule_inputs = ['gsc_impressions_avg', 'position_avg']
leakage_cols = ['trend_pct', 'is_declining_label']

top20['ctr'] = top20['gsc_clicks_avg'] / top20['gsc_impressions_avg']
low_ctr_count = (top20['ctr'] < 0.001).sum()
print(f"{low_ctr_count} of 20 top-ranked pages have CTR below 0.001 (near-zero clicks)")
print("\nRule inputs used:", rule_inputs)
print("Confirmed NOT used as rule inputs:", leakage_cols)

,content_id,gsc_impressions_avg,position_avg,weak_pick_flag
0,content_e8a52cf3d5988c07,9544.866667,16.018687,no flag
1,content_36e53e9c707674fc,7327.266667,33.354423,no flag
2,content_3df3f32f3fd58dea,5602.733333,24.501281,no flag
3,content_5e1c049f62e33b11,4862.666667,18.233335,no flag
4,content_82e35c4845e6c391,4677.933333,18.269589,near-zero CTR despite high impressions (possib...
5,content_df47d1b976106de4,4422.800000,25.221991,no flag
6,content_559cdd76da9306de,4161.200000,37.866104,near-zero CTR despite high impressions (possib...
7,content_bdf60c86117079be,4018.466667,31.046299,near-zero CTR despite high impressions (possib...
8,content_9fff53e827550f9d,3764.666667,22.357092,no flag
9,content_573804af4f4fa09f,3529.866667,29.926699,near-zero CTR despite high impressions (possib...


12 of 20 top-ranked pages have CTR below 0.001 (near-zero clicks)

Rule inputs used: ['gsc_impressions_avg', 'position_avg']
Confirmed NOT used as rule inputs: ['trend_pct', 'is_declining_label']


In [11]:
zero_position_count = (df['position_avg'] == 0).sum()
print(f"Rows with position_avg == 0 (possible 'no data' placeholder): {zero_position_count}")

Rows with position_avg == 0 (possible 'no data' placeholder): 0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.